In [8]:
!pip install pandas
!pip install openpyxl


[notice] A new release of pip is available: 24.0 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)



[notice] A new release of pip is available: 24.0 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [58]:
from groq import Groq
import os
from dotenv import load_dotenv
import pandas as pd
import json

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY"),
)


In [62]:
def ask_groq(question):
    try:
        response = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": question,
                }
            ],
            model="llama-3.3-70b-specdec",
        )
        if 'error' in response:
            return {'error': {'message': response['error']['message'], 'type': response['error'].get('type', ''), 'code': response['error'].get('code', '')}}
        return response.choices[0].message.content
    except Exception as e:
        print(str(e))
        try:
            error_info = json.loads(str(e).split(" - ", 1)[1])
            return {"code": error_info['error']['code'], "message": error_info['error']['message']}
        except (json.JSONDecodeError, IndexError, KeyError):
            return {"code": "unknown_error", "message": str(e)}

In [57]:
df = pd.read_excel("static/LEVANTAMENTO_COLUNAS_TABELAS_GERENCIAL.xlsx", dtype={'TABELA': 'object', 'COLUNA': 'object', 'TIPOLOGIA': 'object', 'DESCRIÇÃO': 'object'})
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2393 entries, 0 to 2392
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   TABELA     2393 non-null   object
 1   COLUNA     2393 non-null   object
 2   TIPOLOGIA  2393 non-null   object
 3   DESCRIÇÃO  0 non-null      object
dtypes: object(4)
memory usage: 74.9+ KB


In [63]:
row = df.head(1)
response = ask_groq(f"Chat, se você estivesse fazendo um trabalho de documentação de um banco de dados e precisasse descrever em poucas palavras (no máximo 30) a coluna {row['COLUNA']}, do tipo {row['TIPOLOGIA']}, que pertênce a tabela {row['TABELA']}, como você descreveria? Por favor simplesmente responda com o texto da descrição entre asteriscos duplos (**texto**) para que eu identifique e utilize de forma mais fácil sua resposta.             Obs: Se as primeiras letras da descrição da coluna, que antecedem um underline (_), tiverem um significado de sufixo, por exemplo: CD - Código; FL - Flag; DT - Data; DS - Descrição; OCC - Ocorrencia; CRED - Credenciamento; OS - Ordem de serviço; POS - Terminal de vendas; TEF - Terminal de vendas; etc, por favor, utilize este significado para descrever a coluna, por exemplo:  FL_DEBITO_AUTORIZADO, Descrição: **Flag que sinaliza um débito Autorizado** ",)
response

'**Código do Grupo de Usuário**'

In [33]:
def safe_ask_groq(question, row):
    try:
        return ask_groq(question)
    except Exception as e:
        print(f"Error: {e}")
        print(f"Coluna: {row['COLUNA']}, Descrição: not saved")
        return "Service Unavailable"

for index, row in df.iterrows():
    if pd.isna(row['DESCRIÇÃO']):
        description = safe_ask_groq(
            f"Chat, se você estivesse fazendo um trabalho de documentação de um banco de dados e precisasse descrever em poucas palavras (no máximo 30) a coluna {row['COLUNA']}, do tipo {row['TIPOLOGIA']}, que pertênce a tabela {row['TABELA']}, como você descreveria? Por favor simplesmente responda com o texto da descrição entre asteriscos duplos (**texto**) para que eu identifique e utilize de forma mais fácil sua resposta.             Obs: Se as primeiras letras da descrição da coluna, que antecedem um underline (_), tiverem um significado de sufixo, por exemplo: CD - Código; FL - Flag; DT - Data; DS - Descrição; OCC - Ocorrencia; CRED - Credenciamento; OS - Ordem de serviço; POS - Terminal de vendas; TEF - Terminal de vendas; etc, por favor, utilize este significado para descrever a coluna, por exemplo:  FL_DEBITO_AUTORIZADO, Descrição: **Flag que sinaliza um débito Autorizado** ",
            row
        )
        df.at[index, 'DESCRIÇÃO'] = description
        print(f"Coluna: {row['COLUNA']}, Descrição: {description}")

Coluna: CD_GRUPO_USUARIO, Descrição: **Código do Grupo de Usuário**
Coluna: CD_TIPO_OCORRENCIA, Descrição: **Código do Tipo de Ocorrência**
Coluna: CD_STATUS, Descrição: **Código de Status**
Coluna: FL_ACAO, Descrição: **Flag de Ação**
Coluna: DT_PROCESSADO, Descrição: **Data em que o processo foi executado**
Coluna: FL_ACAO, Descrição: **Flag que sinaliza uma ação**
Coluna: NOME_TABELA, Descrição: **Nome da tabela de agenda**
Coluna: CD_ESTAB_NEGOCIACAO, Descrição: **Código de estabelecimento de negociação**
Coluna: DT_PROCESSADO, Descrição: **Data em que o processo foi realizado**
Coluna: CD_TIPO_CONEXAO, Descrição: **Código do tipo de conexão**
Coluna: CD_LOG_ARQ_ALTERACAO_TAXAS, Descrição: **Código do Log de Alteração de Taxas**
Coluna: DT_GERACAO, Descrição: **Data de geração do arquivo**
Coluna: DS_ATRIBUTO, Descrição: **Descrição do Atributo**
Coluna: SG_ATRIBUTO, Descrição: **SG de Atributo**
Coluna: DS_CAMPO_TABELA_DE_PARA, Descrição: **Descrição da tabela de para**
Coluna: DS

In [28]:
# df['DESCRIÇÃO'] = df['DESCRIÇÃO'].apply(lambda x: None)

In [38]:
# Contar quantas linhas da coluna DESCRIÇÃO estão vazias
empty_count = df['DESCRIÇÃO'].str.contains('not saved').sum()

# Calcular o percentual de preenchimento
total_count = len(df)
filled_count = total_count - empty_count
filled_percentage = (filled_count / total_count) * 100

print(f"Linhas vazias na coluna DESCRIÇÃO: {empty_count}")
print(f"Percentual de preenchimento: {filled_percentage:.2f}%")
# Mostrar os valores que mais se repetem na coluna DESCRIÇÃO
most_common_descriptions = df['DESCRIÇÃO'].value_counts().head(10)
print("Valores que mais se repetem na coluna DESCRIÇÃO:")
print(most_common_descriptions)

Linhas vazias na coluna DESCRIÇÃO: 0
Percentual de preenchimento: 100.00%
Valores que mais se repetem na coluna DESCRIÇÃO:
DESCRIÇÃO
Service Unavailable                     2107
**Código do Coordenador Regional**         3
**Código da Regional**                     3
**Data da Exportação**                     3
**Código de Status**                       2
**Código do Estoque**                      2
**Código da bandeira**                     2
**Flag que indica posição de venda**       2
**Código do tipo de cartão**               2
**Código do Supervisor**                   2
Name: count, dtype: int64


In [ ]:
df.to_excel("static/LEVANTAMENTO_COLUNAS_TABELAS_GERENCIAL_ATUALIZADO.xlsx", index=False)